## 6.2 边信息该进 Value，还是进 $QK^T$ 的广播矩阵

**用户反馈**：上一轮 6.1 已完成 GET-Zero × T(R,O) 的论文与代码精读。本轮聚焦一个精确架构问题：`指尖的信息，边信息是融入 value，还是 $QK^T$ 的广播矩阵（不是直接的注意力分数）`。用户补充：这里说的“广播矩阵”更接近 `TRO-Grasp/model/denoiser.py` 里 `GraphLayer.forward()` 的逐元素 `Q \odot K` + 分组 broadcast，而不是 GET-Zero 的 Graphormer 式标量 bias。

**分析**：

### 一、先把三种注入机制严格区分

| 机制 | 数学形式 | 代表 | 作用位置 | 实际含义 |
|---|---|---|---|---|
| **标量 bias on score** | $A_{ij} = \frac{Q_iK_j^\top}{\sqrt d} + b_{ij}$ | GET-Zero | score | 只改“看谁” |
| **broadcast gate on $QK$** | $G_{ijg} = \operatorname{softmax}_j(\mathrm{GroupSum}(Q_i \odot K_j \odot m_{ij}))$ | 你说的“广播矩阵” | score 前的分组门控 | 不是标量，而是每组/每通道地改“看谁、看多少” |
| **edge-conditioned value** | $\tilde V_{ij} = h_V(h_i, h_j, e_{ij})$ | TRO-Grasp / 当前 `plan.ipynb` 草稿 | value / message | 改“传什么” |

这里第 2 类要特别强调：它**不是** GET-Zero 那种“每头一个标量 bias”，而是先形成一个 $(i,j,g)$ 级别的 gate，再广播回 feature channels。因此它更像**向量/分组级路由**，不是纯 scalar attention score。

### 二、GET-Zero 的实际做法：只有标量 bias，没有 broadcast gate，也没有 edge-conditioned value

GET 论文 Sec. III-B（正文第 3 页）明确写的是：
$$
A_{ij} = \frac{Q_iK_j^\top}{\sqrt d} + s_{\phi^{SPD}(i,j)} + p_{\phi^P(i,j)} + c_{\phi^C(i,j)}.
$$

代码也完全一致：
- `get_zero/get_zero/distill/models/embodiment_attention.py:382-477`：`_sa_block()` 先构造 `attention_bias`；
- `...:477`：把 `attention_bias` 送入 `self.self_attn(...)`；
- `...:920-930`：bias 只是在 softmax 前/后加到 `attn_output_weights`；
- `...:935`：最后仍然是 `attn_output = torch.bmm(attn_output_weights, v)`。

所以 GET 的 message 仍然是
$$
h_i' = \sum_j \alpha_{ij} V_j,
$$
其中 $V_j$ **不带边信息**。它的图结构只影响 routing，不影响 content。

### 三、TRO-Grasp 的实际做法：代码比论文更接近“broadcast gate + edge-conditioned value”

T(R,O) 论文 Eq. (12)（正文第 5 页）写的是：
$$
Q = h_Q(X_{tgt}), \qquad K = h_K(X_{src}), \qquad V = h_V(\mathrm{concat}(X_{tgt}, X_{src}, E)).
$$
这已经说明：**edge 在 value 里，不在论文公式的 $Q,K$ 里**。

但代码里还有一个比论文写得更细的实现特征：
- `TRO-Grasp/model/denoiser.py:244-248`：`or_attn = or_query * or_key` 后 reshape、group-sum、softmax、再 expand 回通道；
- `...:255-256`：`or_value = self.or_value_fc(torch.cat([or_edge_o, or_edge_r, or_edge_f], dim=-1))`，然后 `agg_or = (or_attn * or_value).sum(2)`；
- `...:283-287`、`...:294-295`：RR attention 完全同构。

这意味着 TRO 真正的实现不是标准单标量
$$
\alpha_{ij} = \operatorname{softmax}_j(Q_iK_j^\top),
$$
而更像
$$
M_{ijc} = Q_{ic} K_{jc},
$$
$$
G_{ijg} = \operatorname{softmax}_j\!\big(\mathrm{GroupSum}(M_{ij,:})\big),
$$
$$
\Gamma_{ijc} = \mathrm{Broadcast}(G_{ijg}),
$$
$$
h_i' = \sum_j \Gamma_{ijc} \odot \tilde V_{ijc},
\qquad
\tilde V_{ij} = h_V(h_i, h_j, e_{ij}).
$$

所以 **TRO 其实同时具有**：
1. 一个 **group-wise broadcast gate**（来自 $Q \odot K$）；
2. 一个 **edge-conditioned value**（来自 $h_V(\cdot, e_{ij})$）。

只是要注意：**边信息只进入第 2 项，没有进入第 1 项。**

### 四、`plan.ipynb` 当前草稿其实已经站在“TRO 主线”上，而不是“broadcast matrix 主线”

`plan.ipynb` §3.4（`AnyMani/source/anymani/ideas/graph/plan.ipynb:176-209`）当前写的是：
$$
\alpha_{ij}^{(\ell)} \propto \exp\left(\frac{Q_iK_j^\top}{\sqrt d} + b_{ij}^{topo} + b_{ij}^{geom}\right),
$$
$$
\tilde V_{ij}^{(\ell)} = h_\psi\big(h_i^{(\ell)}, h_j^{(\ell)}, e_{ij}\big),
$$
$$
\hat h_i^{(\ell)} = \sum_j \alpha_{ij}^{(\ell)} \tilde V_{ij}^{(\ell)}.
$$

这说明当前 draft 的默认立场是：
- **routing**：GET-style 标量 bias；
- **content**：TRO-style edge-conditioned value。

也就是说，当前 plan **还没有真正引入**你说的“$QK^T$ 的广播矩阵”这一条分支。它现在更像“GET 的 score + TRO 的 value”，不是“TRO 代码那种 grouped gate”。

### 五、你说的“广播矩阵”如果要严格写清楚，最合理的数学形式应是下面这个第三分支

如果我们真想把边信息注入到“不是直接标量 score，而是 $QK^T$ 的 broadcast matrix”，更准确的写法应是：
$$
Q_i = W_Q h_i,\qquad K_j = W_K h_j,\qquad g_{ij} = h_G(e_{ij}),
$$
先形成逐通道或逐组的乘性门控：
$$
M_{ijc} = Q_{ic}\, K_{jc}\, g_{ijc},
$$
再按组聚合成 broadcast gate：
$$
G_{ijg} = \operatorname{softmax}_j\!\big(\mathrm{GroupSum}(M_{ij,:})\big),
$$
最后广播回通道去加权 value：
$$
\Gamma_{ijc} = \mathrm{Broadcast}(G_{ijg}),
\qquad
h_i' = \sum_j \Gamma_{ijc} \odot \tilde V_{ijc}.
$$

这里要注意：
- 若 $\tilde V_{ij}=V_j$，那么 edge 只影响 routing，不影响 content；
- 若 $\tilde V_{ij}=h_V(h_i,h_j,e_{ij})$，那么 edge 同时影响 routing 与 content。

因此，“edge 进 broadcast matrix”本质上是在增强 **who-to-talk-to**；
“edge 进 value”本质上是在增强 **what-to-say**。

### 六、对“指尖信息”而言，这两条路的职责并不对称

我觉得这是本轮最关键的判断。

#### 1. 指尖首先是 **局部接触执行器**，不是普通中间关节

指尖（fingertip / tip-head）和中间关节最大的不同是：
- 它是**末端执行器**，很多任务相关量在它这里闭环；
- 它承载的是**接触面/接触法向/接触半径/触觉覆盖**这类局部语义；
- 真正的接触点往往**不等于 distal joint/link frame**。

你在 `mine.ipynb:102-108` 已经写了一个很关键的工程直觉：想用固定虚拟关节/虚拟 link（如 `index_tip_head`）把**真实接触点坐标系**显式挂到 `fingertip` 下。这件事在方法上是有价值的，因为它把“指尖接触偏移”从一个隐含常数，变成了一个**可显式建模的局部末端节点**。

#### 2. 因此，很多“指尖信息”本质上更像 **node-local / message content**，不是纯 routing bias

例如下面这些信息，更适合进 **node/value**：
- 指尖接触面半径、曲率、shape type（flat / rounded / hemisphere）；
- distal link 到真实接触点 frame 的固定偏移 $\Delta T_{link\rightarrow tip}$；
- tactile availability / tactile resolution；
- tip local normal / principal axis；
- thumb tip vs normal finger tip 的局部几何差异。

因为这些量影响的是：

> 当某个 tip 节点把信息传给别的节点时，它传出的**消息内容本身**应该不同。

这类差异如果只放进 broadcast matrix，只能让网络学到“tip 更该关注谁”，但很难表达“tip 传出去的内容和普通关节根本不是一类语义”。

#### 3. 但有一类指尖信息确实更像 routing / gating

例如：
- tip-tip / tip-thumb opposition 的重要性；
- same-finger vs cross-finger vs thumb-related 的通信优先级；
- 哪些 pair 更可能构成稳定 pinch / enclosure；
- terminal node 对 palm 或另一 tip 的 reachability saliency。

这类量更像：

> 在当前 query 节点看来，哪些 neighbor 的信息应该被优先读入？

它更适合进入 **QK broadcast gate / bias**，因为它本质上决定“这条边该不该被放大”。

### 七、因此如果只能二选一，我更倾向：**指尖相关连续几何先优先进 value，不优先进 broadcast matrix**

理由有三条：

1. **手内操作里，指尖差异首先是 contact semantics，不只是拓扑 saliency**；
2. **GET 已经覆盖了“看谁”的最弱版本**，即使只保留 bias，也并非完全没有 routing；
3. **TRO 论文与代码都站在这条线上**：论文 Eq. (12) 把 edge 放在 $V$，代码 `or_value/rr_value` 也明确如此。

所以如果第一版要稳，我会建议：

> **主路：edge-conditioned value；辅路：tip-aware scalar/group gate（可选）。**

### 八、一个更稳的折中版本：不是“只选一个”，而是按职责拆成两路

#### 路 1：连续几何 / 指尖局部语义 → value
$$
\tilde V_{ij} = h_V\big(h_i, h_j, e_{ij}^{geom}, u_i^{tip}, u_j^{tip}\big)
$$

其中 $u_i^{tip}$ 是 tip-local descriptor，例如：
- `is_tip`；
- `tip_type`；
- `contact_patch_radius`；
- `\Delta T_{link\rightarrow tip}`；
- `thumb_flag`。

#### 路 2：tip-related saliency → broadcast gate / bias
$$
G_{ijg} = \operatorname{softmax}_j\!\big(\mathrm{GroupSum}(Q_i \odot K_j \odot (1 + h_G(e_{ij}^{tip})) )\big)
$$

其中 $e_{ij}^{tip}$ 只放**轻量的 pairwise saliency**，例如：
- `is_tip_i`, `is_tip_j`；
- `is_thumb_tip_pair`；
- `same_finger / cross_finger`；
- `tip-tip rest distance`；
- `opposition prior`。

这样做的好处是：
- **value 路**负责接触语义；
- **gate 路**负责通信优先级；
- 不会把所有 tip 信息都粗暴塞到单一路径里。

### 九、落回当前 paper/story，我觉得最合理的第一版拍板是

如果只考虑第一篇 paper 的实现稳健性和叙事清晰度，我建议当前版本先写成：
$$
\alpha_{ij} \propto \exp\left(\frac{Q_iK_j^\top}{\sqrt d} + b_{ij}^{topo} + b_{ij}^{tip-lite}\right),
$$
$$
\tilde V_{ij} = h_V(h_i, h_j, e_{ij}^{geom}, u_i^{tip}, u_j^{tip}),
$$
$$
h_i' = \sum_j \alpha_{ij} \, \tilde V_{ij}.
$$

也就是：
- **不马上上完整 TRO-style grouped broadcast gate**；
- 先保留 GET-style score path；
- 把**真正重的指尖/连续几何信息优先放进 value**；
- 若后续实验发现“routing 还不够”，再把 grouped broadcast gate 作为下一层升级。

这比一开始就把 edge 硬塞进 broadcast matrix 更稳，因为那样会同时改 attention 机制和 relation 表达，归因会变脏。

### 十、对图结构本身的一个补充判断：tip 最好单独显式化

如果你准备认真建模“指尖信息”，我认为有一个比“进 value 还是进 gate”更上游、也更应该尽早拍板的点：

> **要不要把实际 contact point 用固定 tip-head 节点显式化？**

如果保留当前 distal link 作为 tip 代理，那么很多真正的 tip 信息都要靠额外字段补丁描述；
而如果像 `mine.ipynb:104-108` 那样加一个 fixed `tip_head`，则：
- node-local tip descriptor 更自然；
- tip-related pairwise edge 也更明确；
- 后续若加 tactile / contact state，也更容易挂接。

这件事对“指尖信息到底进哪里”会有直接影响。

**小结**：GET-Zero 是 **scalar bias only**；TRO-Grasp 代码更像 **group-wise broadcast gate + edge-conditioned value**，但**edge 只进 value**；`plan.ipynb` 当前默认也是 **bias on score + edge-conditioned value**；对“指尖信息”而言，我更倾向 **连续几何/接触语义优先进 value**，而把轻量的 tip saliency 放进 score / gate；如果第一版只能拍板一个主路径，我建议 **先选 value，不先选 broadcast matrix**；但在这之前，更值得先确认的是：**是否显式引入 fixed tip-head / virtual tip node**。

**待确认**：下一步你更想先拍板哪一个层级？
1. `主注入位置`：先确认“tip 信息主路进 value”；
2. `broadcast gate 版本`：先确认是否要做 TRO-like grouped gate；
3. `双通路折中`：tip-local 进 value，tip-pair saliency 进 gate / bias；
4. `先定图结构`：先确认是否引入 fixed `tip_head` / virtual tip node，再谈注入位置。